In [ ]:
# colab
# !uv pip install -qqq lime

In [ ]:
from collections import Counter

import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
from datasets import load_dataset
from IPython.display import HTML
from lime.lime_text import LimeTextExplainer
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report

In [ ]:
seed = 42
y_col_name = "label"

model_name_st = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

## 1. データの読み込み

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")

In [ ]:
dataset["train"]

In [ ]:
# 動作確認用
df_train = dataset["train"][:100]
df_test = dataset["test"][:100]

# 本番用
# df_train = dataset["train"].to_pandas()
# df_test = dataset["test"].to_pandas()

In [ ]:
df_test["text"]

## 2. モデルの設定、学習

### 2.1 学習

In [ ]:
# embedding用のモデルを設定

# 動作確認
model_st = SentenceTransformer(model_name_st)

# 本番用
# model_st = SentenceTransformer(
#     model_name_st,
#     device="cuda"
# )

In [ ]:

print("train data encoding starts")
X_train_encoded = model_st.encode(
    df_train["text"], # .to_pandas()でdf_trainを作った場合seriesになってるのでつける→.to_numpy()
    batch_size=32,            # 一度に処理する件数を32件に制限（メモリ対策）
    show_progress_bar=True, #進捗barの表示
)

print("test data encoding starts")
X_test_encoded = model_st.encode(
    df_test["text"], # .to_pandas()でdf_trainを作った場合はつける→.to_numpy()
    batch_size=32,            # 一度に処理する件数を32件に制限（メモリ対策）
    show_progress_bar=True, #進捗barの表示
)

In [ ]:
# 分類器のlightgbmの設定

model_lgb = lgb.LGBMClassifier(
    verbose = -1,
    random_state=seed
)

In [ ]:
# lightgbmの学習
model_lgb.fit(X_train_encoded, df_train[y_col_name])

### 2.2 評価

In [ ]:
y_pred = model_lgb.predict(X_test_encoded)

In [ ]:
dict_report = classification_report(df_test[y_col_name], y_pred, output_dict=True)

In [ ]:
pd.DataFrame(dict_report).T
# 一定の精度があることを確認してから、limeに進む

## 3.LIMEで単語の影響度を確認

In [ ]:
# LIME用のパイプライン（ベクトル化＋予測確率の計算）を作成

def predict_pipeline(texts):
    vectors = model_st.encode(texts)
  
    return model_lgb.predict_proba(vectors)

In [ ]:
explainer = LimeTextExplainer(class_names=["ネガティブ", "ポジティブ"]) # 0がネガティブ、1がポジティブと設定

In [ ]:
# テキストを１サンプル指定して、解釈

target_text = "The screenplay of this movie is exceptionally detailed. It is a remarkable masterpiece that will go down in history!"
# target_text = "This film is so riddled with plot holes that it's unwatchable. A complete waste of funding."
# target_text = "The movie runs for two hours and covers the historical events of the early 20th century."

exp = explainer.explain_instance(target_text, predict_pipeline, num_features=5)

In [ ]:
html_data = exp.as_html()
display(HTML(html_data))